# Lab 28 — Add Retry/Fallback Strategies to the ClaimsIQ Agent Flow

**Module:** Performance Optimization · Day 15 · Session 03
**Duration:** ~35-45 minutes

### What you will do
Wrap ClaimsIQ's fraud-signals check with retries, a circuit breaker, and
a safe ESCALATED fallback — then simulate a real Snowflake outage and
confirm the system degrades gracefully instead of crashing.

### Prerequisite
ClaimsIQ Notebooks 00-01 must have run already.

## Step 1 — Reconnect to the MCP server

In [ ]:
%pip install -q snowflake-connector-python

In [ ]:
import time, random
from mcp_snowflake_server import claims_server, SimpleMCPClient

mcp_client = SimpleMCPClient(claims_server)
mcp_client.connect()
print("Connected.")

## Step 2 — Write `call_with_retry()` with exponential backoff

In [ ]:
def call_with_retry(fn, max_attempts=3, base_delay=1, *args, **kwargs):
    for attempt in range(max_attempts):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            print(f"  [attempt {attempt + 1}/{max_attempts}] failed: {e}")
            if attempt == max_attempts - 1:
                raise
            wait = base_delay * (2 ** attempt)
            print(f"  retrying in {wait}s...")
            time.sleep(wait)

print("call_with_retry() ready.")

## Step 3 — Write a minimal `CircuitBreaker` class

In [ ]:
class CircuitBreaker:
    def __init__(self, failure_threshold=3, cooldown_seconds=10):
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.failure_count = 0
        self.state = "CLOSED"
        self.opened_at = None

    def is_open(self) -> bool:
        if self.state == "OPEN":
            if time.time() - self.opened_at >= self.cooldown_seconds:
                self.state = "HALF-OPEN"
                print("  [circuit breaker] HALF-OPEN — will test the next call")
                return False
            return True
        return False

    def record_success(self):
        if self.state == "HALF-OPEN":
            print("  [circuit breaker] recovered -> CLOSED")
        self.state = "CLOSED"
        self.failure_count = 0

    def record_failure(self):
        self.failure_count += 1
        if self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()
            print(f"  [circuit breaker] {self.failure_count} failures -> OPEN")

circuit_breaker = CircuitBreaker(failure_threshold=3, cooldown_seconds=8)
print("CircuitBreaker ready.")

## Step 4 — Wrap `check_fraud_signals` as `resilient_fraud_check()`

In [ ]:
def resilient_fraud_check(customer_id: str) -> dict:
    if circuit_breaker.is_open():
        return {"status": "ESCALATED", "reason": "Fraud check unavailable (circuit open), escalating for safety."}
    try:
        result = call_with_retry(
            mcp_client.call_tool, max_attempts=3, base_delay=1,
            name="check_fraud_signals", customer_id=customer_id,
        )
        circuit_breaker.record_success()
        return {"status": "OK", "signals": result}
    except Exception:
        circuit_breaker.record_failure()
        return {"status": "ESCALATED", "reason": "Fraud check failed after retries, escalating for safety."}

print("resilient_fraud_check() ready.")

## Step 5 — Run a normal case, confirm it still works

In [ ]:
result = resilient_fraud_check("CUST99001")
print(result)

## Step 6 — Simulate a failure

Temporarily replace `mcp_client.call_tool` with a function that always
raises an exception, simulating a genuine Snowflake outage.

In [ ]:
_real_call_tool = mcp_client.call_tool

def broken_call_tool(name, **kwargs):
    raise ConnectionError("Simulated Snowflake outage")

mcp_client.call_tool = broken_call_tool
print("Snowflake calls will now fail (simulated outage).")

## Step 7 — Confirm 3 retries happen, then it fails over to ESCALATED

In [ ]:
result = resilient_fraud_check("CUST99002")
print("\nFinal result:", result)
assert result["status"] == "ESCALATED", "Expected a graceful ESCALATED fallback, not a crash!"
print("PASS — the system degraded gracefully instead of crashing.")

## Step 8 — Confirm the circuit breaker opens after repeated failures

In [ ]:
# Run a few more failing calls to push past the failure threshold
for i in range(3):
    print(f"\n--- call {i+1} ---")
    resilient_fraud_check(f"CUST9900{i}")

print(f"\nCircuit breaker state: {circuit_breaker.state}")
assert circuit_breaker.state == "OPEN", "Circuit breaker should be OPEN after repeated failures!"
print("PASS — circuit breaker opened, further calls will fail fast without even trying Snowflake.")

## Step 9 — Restore the real connection, confirm recovery

In [ ]:
mcp_client.call_tool = _real_call_tool
print("Real Snowflake connection restored.")
print(f"Circuit breaker state before cooldown: {circuit_breaker.state}")

print("\nWaiting for the cooldown period...")
time.sleep(9)

result = resilient_fraud_check("CUST99001")
print("\nResult after cooldown + real connection restored:", result)
print(f"Circuit breaker state: {circuit_breaker.state}")

## Deliverable

1. The full output from Steps 6-8, showing the retries, the ESCALATED
   fallback, and the circuit breaker opening.
2. Step 9's recovery confirmation.
3. One paragraph: `resilient_fraud_check()` falls back to `ESCALATED`,
   never to `APPROVED` or `DENIED`. Referring to Session 03's slides on
   fallback anti-patterns, explain in your own words why this specific
   choice matters for a claims system, and what could go wrong if the
   fallback were `APPROVED` instead.
4. (Stretch) `call_with_retry` currently retries EVERY exception
   identically. Real Snowflake errors include some that are worth
   retrying (a transient timeout) and some that never will succeed no
   matter how many times you retry (a permissions error, like the ones
   from your own earlier setup). Modify `call_with_retry` to stop
   retrying immediately on a non-transient error type.